# P18 – AI Patient Status Assignment Tool: Data Access Demo

This notebook downloads every **openly accessible** dataset listed in *P18_Project_Ideas_and_Datasets.docx* and shows the fields each project idea depends on. Run it top to bottom. Files are cached in `./data/`, so later runs are fast.

| Section | Source | Doc idea |
|---|---|---|
| 1a–1c | MIMIC-IV Demo + MIMIC-IV-ED Demo: status labels and an ED cohort | 1. Early Status Predictor |
| 1d–1e | MIMIC-IV Demo: status order sequence, timestamped labs, ED vitals | 4. Conversion Monitor |
| 1f | MIMIC-IV Demo: a patient fact sheet for building labeled test notes | 2–3. Status Review / Gap Coach |
| 1g | MIMIC-IV Demo: discharge destination (SNF) | 5. Patient Impact Simulator |
| 2 | NHAMCS 2019 ED public-use file (CDC) | 1. Early Status Predictor |
| 3 | MTSamples (Kaggle, CC0) | 2–3 |
| 4 | Asclepius synthetic notes (Hugging Face) | 2–3 |
| 5 | CMS rules: 42 CFR 412.3, Benefit Policy Manual Ch. 1, Inpatient Only list; ICD-10-CM | 2–3 (rules), 5 |
| 6 | Full MIMIC-IV / MIMIC-IV-Note (credentialed): how to plug it in | All |

> **Data rules:** keep `data/` out of GitHub (add it to `.gitignore`). Open demo data may be used with cloud LLMs. Credentialed MIMIC data may **not** (see Section 6).

Requirements: `pandas`, `requests` (both included in Anaconda).

In [1]:
import re
import zipfile
from pathlib import Path

import pandas as pd
import requests

DATA = Path("data")
DATA.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 100)
STATUS = []  # one row per dataset, summarized at the end


def fetch(url, name, **kwargs):
    """Download url to data/name once; reuse the cached copy afterwards."""
    path = DATA / name
    if not path.exists():
        r = requests.get(url, timeout=300, **kwargs)
        r.raise_for_status()
        path.write_bytes(r.content)
    return path


def log(dataset, access, result):
    STATUS.append({"dataset": dataset, "access": access, "result": result})
    print(f"OK  {dataset}: {result}")

## 1. MIMIC-IV Demo + MIMIC-IV-ED Demo (open access)

100 real de-identified patients from Beth Israel Deaconess Medical Center. No account is needed for the demo. The full versions use **exactly the same tables and columns**, so code written here runs unchanged on full MIMIC-IV.

In [2]:
MIMIC = "https://physionet.org/files/mimic-iv-demo/2.2/hosp/"
MIMIC_ED = "https://physionet.org/files/mimic-iv-ed-demo/2.2/ed/"

admissions = pd.read_csv(fetch(MIMIC + "admissions.csv.gz", "admissions.csv.gz"))
poe = pd.read_csv(fetch(MIMIC + "poe.csv.gz", "poe.csv.gz"),
                  usecols=["poe_id", "subject_id", "hadm_id", "ordertime", "order_type"])
poe_detail = pd.read_csv(fetch(MIMIC + "poe_detail.csv.gz", "poe_detail.csv.gz"))
labevents = pd.read_csv(fetch(MIMIC + "labevents.csv.gz", "labevents.csv.gz"),
                        usecols=["subject_id", "hadm_id", "itemid", "charttime", "valuenum", "valueuom", "flag"])
d_labitems = pd.read_csv(fetch(MIMIC + "d_labitems.csv.gz", "d_labitems.csv.gz"))
transfers = pd.read_csv(fetch(MIMIC + "transfers.csv.gz", "transfers.csv.gz"))
diagnoses = pd.read_csv(fetch(MIMIC + "diagnoses_icd.csv.gz", "diagnoses_icd.csv.gz"))
d_icd = pd.read_csv(fetch(MIMIC + "d_icd_diagnoses.csv.gz", "d_icd_diagnoses.csv.gz"))
icustays = pd.read_csv(fetch("https://physionet.org/files/mimic-iv-demo/2.2/icu/icustays.csv.gz", "icustays.csv.gz"))

edstays = pd.read_csv(fetch(MIMIC_ED + "edstays.csv.gz", "edstays.csv.gz"))
triage = pd.read_csv(fetch(MIMIC_ED + "triage.csv.gz", "triage.csv.gz"))
ed_vitals = pd.read_csv(fetch(MIMIC_ED + "vitalsign.csv.gz", "ed_vitalsign.csv.gz"))

log("MIMIC-IV Demo", "open", f"{admissions.subject_id.nunique()} patients, {len(admissions)} hospital admissions, "
    f"{len(labevents):,} lab results")
log("MIMIC-IV-ED Demo", "open", f"{len(edstays)} ED stays, {len(triage)} triage records, {len(ed_vitals):,} ED vital-sign rows")

OK  MIMIC-IV Demo: 100 patients, 275 hospital admissions, 107,727 lab results
OK  MIMIC-IV-ED Demo: 222 ED stays, 222 triage records, 1,038 ED vital-sign rows


### 1a. Status labels, source 1: `admissions.admission_type`
Observation types are recorded separately from inpatient types.

In [3]:
OBS_TYPES = {"OBSERVATION ADMIT", "EU OBSERVATION", "DIRECT OBSERVATION", "AMBULATORY OBSERVATION"}
admissions["status_from_type"] = admissions.admission_type.map(
    lambda t: "observation" if t in OBS_TYPES else "inpatient")
admissions.groupby(["status_from_type", "admission_type"]).size().rename("admissions").to_frame()

admissions
status_from_type admission_type                         
inpatient        DIRECT EMER.                         15
                 ELECTIVE                             13
                 EW EMER.                            104
                 SURGICAL SAME DAY ADMISSION          18
                 URGENT                               38
observation      AMBULATORY OBSERVATION                5
                 DIRECT OBSERVATION                    7
                 EU OBSERVATION                       30
                 OBSERVATION ADMIT                    45

### 1b. Status labels, source 2: the physician's order (`poe_detail`, field *Admit category*)
This is the actual status order a physician entered: **"Admit to inpatient"** or **"Place in observation"**.

In [4]:
status_orders = (
    poe_detail[poe_detail.field_name == "Admit category"]
    .merge(poe[["poe_id", "hadm_id", "ordertime"]], on="poe_id")
    .dropna(subset=["hadm_id"])
    .sort_values("ordertime")
)
status_orders.field_value.value_counts().rename("orders").to_frame()

,orders
field_value,
Admit to inpatient,876
Place in observation,207
Place in PACU,1


### 1c. ED-visit cohort with a three-class label (Idea 1)
Each ED stay is labeled **discharge**, **observation**, or **inpatient**, then joined to triage features. The crosstab checks the two label sources against each other. Disagreements are worth discussing with Kevin.

In [5]:
first_order = status_orders.groupby("hadm_id").field_value.first()

ed = edstays.merge(admissions[["hadm_id", "admission_type", "status_from_type"]], on="hadm_id", how="left")
ed["first_status_order"] = ed.hadm_id.map(first_order)
ed["label"] = ed.status_from_type.where(ed.disposition == "ADMITTED")
ed.loc[ed.disposition == "HOME", "label"] = "discharge"

print(ed.label.value_counts(dropna=False).to_string(), "\n")
pd.crosstab(ed.status_from_type, ed.first_status_order, dropna=False)

label
inpatient      102
discharge       60
observation     48
NaN             12 



first_status_order,Admit to inpatient,Place in PACU,Place in observation,NaN
status_from_type,,,,
inpatient,66,0,37,1
observation,10,1,56,1
NaN,0,0,0,50


**Reading the crosstab:** stays whose admission type is *inpatient* but whose first order was *Place in observation* are likely observation-to-inpatient conversions (see 1d). Stays with no label (`NaN`) left the ED another way (transfer, left without being seen, etc.) and are excluded from the cohort.

In [6]:
TRIAGE_FEATURES = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp", "pain", "acuity", "chiefcomplaint"]
cohort = ed.dropna(subset=["label"]).merge(triage, on=["subject_id", "stay_id"], how="left")
cohort[["stay_id", "gender", "arrival_transport"] + TRIAGE_FEATURES + ["label"]].head(8)

,stay_id,gender,arrival_transport,temperature,heartrate,resprate,o2sat,sbp,dbp,pain,acuity,chiefcomplaint,label
0,37887480,F,WALK IN,99.1,90.0,26.0,NaN,86.0,61.0,10,1.0,FEVER,inpatient
1,34176810,F,AMBULANCE,97.7,74.0,20.0,96.0,133.0,65.0,3,2.0,"Syncope, Transfer",observation
2,32103106,F,AMBULANCE,96.2,74.0,18.0,100.0,142.0,75.0,0,2.0,"N/V, Tinnitus",discharge
3,38797992,F,AMBULANCE,99.2,130.0,32.0,94.0,106.0,74.0,NaN,1.0,SOB/ABNL LABS,inpatient
4,33473053,F,AMBULANCE,97.5,118.0,18.0,96.0,100.0,56.0,0,2.0,PE,inpatient
5,35758326,F,UNKNOWN,98.8,72.0,18.0,90.0,98.0,48.0,13,2.0,Abd pain,inpatient
6,32259573,F,WALK IN,98.1,89.0,18.0,96.0,131.0,90.0,10,3.0,"Neck pain, Med refill",discharge
7,33693488,F,WALK IN,97.5,73.0,18.0,100.0,137.0,91.0,0,3.0,Eye pain,discharge


### 1d. Observation-to-inpatient conversions (Idea 4: label)
These are hospital stays whose first status order is *Place in observation* and that later have an *Admit to inpatient* order. The rate below looks high, so check for orders re-entered at transfers before using this as a label.

In [7]:
seq = status_orders.groupby("hadm_id").field_value.agg(list)
starts_obs = seq[seq.str[0] == "Place in observation"]
converted = starts_obs[starts_obs.map(lambda v: "Admit to inpatient" in v)]
print(f"stays starting in observation: {len(starts_obs)}")
print(f"  ...later with an inpatient order: {len(converted)}")
log("MIMIC-IV status labels", "open", f"{len(cohort)} labeled ED stays; {len(converted)} candidate conversions")

stays starting in observation: 122
  ...later with an inpatient order: 87
OK  MIMIC-IV status labels: 210 labeled ED stays; 87 candidate conversions


### 1e. Signals over time for a converted stay (Idea 4: features)
Timestamped labs from `labevents` (lactate, troponin, creatinine, WBC), measured from the first observation order. The doc's caveat applies here: bedside vitals (`chartevents`) exist only for ICU stays, so outside the ICU you work mainly with labs, ED vitals, orders, and `transfers` (unit moves).

In [8]:
KEY_LABS = ["Lactate", "Troponin T", "Creatinine", "White Blood Cells"]
key_items = d_labitems[d_labitems.label.isin(KEY_LABS) & (d_labitems.fluid == "Blood")].set_index("itemid").label

obs_start = (status_orders[status_orders.field_value == "Place in observation"]
             .groupby("hadm_id").ordertime.min())
labs = labevents[labevents.itemid.isin(key_items.index) & labevents.hadm_id.isin(converted.index)].copy()
labs["lab"] = labs.itemid.map(key_items)
labs["hours_from_obs"] = ((pd.to_datetime(labs.charttime) - pd.to_datetime(labs.hadm_id.map(obs_start)))
                          .dt.total_seconds() / 3600).round(1)

example_stay = labs.groupby("hadm_id").size().idxmax()
print(f"hadm_id {int(example_stay)}: key labs relative to the first observation order")
print("unit path:", " -> ".join(transfers[transfers.hadm_id == example_stay].sort_values("intime").careunit.dropna()))
(labs[labs.hadm_id == example_stay]
 .pivot_table(index="hours_from_obs", columns="lab", values="valuenum", aggfunc="first")
 .head(12))

hadm_id 28258130: key labs relative to the first observation order
unit path: Emergency Department -> Medical/Surgical Intensive Care Unit (MICU/SICU) -> Medicine


lab,Creatinine,Lactate,Troponin T,White Blood Cells
hours_from_obs,,,,
0.3,2.2,NaN,NaN,6.7
0.8,NaN,3.4,NaN,NaN
4.9,2.0,NaN,NaN,12.4
5.2,NaN,3.9,NaN,NaN
5.8,NaN,3.6,NaN,NaN
8.1,NaN,3.3,NaN,NaN
10.9,1.9,NaN,NaN,13.7
11.1,NaN,3.4,NaN,NaN
16.9,NaN,NaN,NaN,13.2


In [9]:
icu_share = admissions.hadm_id.isin(icustays.hadm_id).mean()
print(f"ED vital-sign rows: {len(ed_vitals):,} across {ed_vitals.stay_id.nunique()} ED stays")
print(f"Admissions with an ICU stay (i.e. with bedside chartevents): {icu_share:.0%}")

ED vital-sign rows: 1,038 across 206 ED stays
Admissions with an ICU stay (i.e. with bedside chartevents): 47%


### 1f. A patient fact sheet for building labeled test notes (Ideas 2–3)
No public dataset pairs notes with status labels, so the doc proposes generating notes **from real demo patients whose status order is known**. This cell assembles the facts for one labeled ED stay. An LLM can then write a realistic ED note from them, and the true label is used to score the copilot. Demo data is open, so a cloud LLM is allowed here (not for credentialed data).

In [10]:
def fact_sheet(stay):
    t = cohort[cohort.stay_id == stay].iloc[0]
    dx = (diagnoses[diagnoses.hadm_id == t.hadm_id].sort_values("seq_num").head(5)
          .merge(d_icd, on=["icd_code", "icd_version"]).long_title.tolist())
    first_labs = (labevents[(labevents.hadm_id == t.hadm_id) & labevents.itemid.isin(key_items.index)]
                  .sort_values("charttime").drop_duplicates("itemid"))
    labs_txt = ", ".join(f"{key_items[i]} {v:g} {u}" for i, v, u in
                         zip(first_labs.itemid, first_labs.valuenum, first_labs.valueuom) if pd.notna(v))
    facts = (f"Sex {t.gender}; arrival by {t.arrival_transport}. Chief complaint: {t.chiefcomplaint}. "
             f"Triage: T {t.temperature}, HR {t.heartrate}, RR {t.resprate}, SpO2 {t.o2sat}, "
             f"BP {t.sbp}/{t.dbp}, pain {t.pain}, acuity {t.acuity}. First labs: {labs_txt or 'none'}. "
             f"Diagnoses: {'; '.join(dx)}.")
    return facts, t.label

admitted = cohort[cohort.label.isin(["observation", "inpatient"])]
for stay in admitted.groupby("label").stay_id.first():
    facts, label = fact_sheet(stay)
    print(f"TRUE LABEL: {label}\n{facts}\n")

TRUE LABEL: inpatient
Sex F; arrival by WALK IN. Chief complaint: FEVER. Triage: T 99.1, HR 90.0, RR 26.0, SpO2 nan, BP 86.0/61.0, pain 10, acuity 1.0. First labs: Creatinine 0.7 mg/dL, White Blood Cells 23.5 K/uL. Diagnoses: Other postoperative infection; Empyema without mention of fistula; Disruption of external operation (surgical) wound; Other specified forms of effusion, except tuberculous; Cellulitis and abscess of trunk.

TRUE LABEL: observation
Sex F; arrival by AMBULANCE. Chief complaint: Syncope, Transfer. Triage: T 97.7, HR 74.0, RR 20.0, SpO2 96.0, BP 133.0/65.0, pain 3, acuity 2.0. First labs: Creatinine 0.7 mg/dL, White Blood Cells 11.8 K/uL. Diagnoses: Syncope and collapse; Cerebral aneurysm, nonruptured; Multiple fractures of ribs, left side, initial encounter for closed fracture; Collapsed vertebra, not elsewhere classified, thoracic region, initial encounter for fracture; Urinary tract infection, site not specified.



### 1g. Discharge destination (Idea 5: skilled nursing facility need)
Note that many observation stays have no recorded destination (`UNKNOWN`), which limits SNF prediction for exactly the patients Idea 5 cares about. Plan around that.

In [11]:
pd.crosstab(admissions.discharge_location.fillna("UNKNOWN"), admissions.status_from_type)

status_from_type,inpatient,observation
discharge_location,,
ACUTE HOSPITAL,1,0
AGAINST ADVICE,2,2
CHRONIC/LONG TERM ACUTE CARE,8,1
DIED,12,3
HOME,57,15
HOME HEALTH CARE,65,11
HOSPICE,4,1
PSYCH FACILITY,2,0
REHAB,10,3


## 2. NHAMCS 2019 Emergency Department public-use file (CDC)

This is a national sample of US ED visits. The 2019 file has **direct observation outcome variables**:
- `OBSDIS`: admitted to observation unit, then **discharged**
- `OBSHOS`: admitted to observation unit, then **hospitalized**
- `ADMITHOS`: admitted to this hospital
- `OBSSTAY`: minutes in the observation unit

Negative values (e.g. -9, -8) are survey codes for blank or unknown. `PATWT` is the survey weight for national estimates. Check the year's codebook before modeling.

In [12]:
nh_zip = fetch("https://ftp.cdc.gov/pub/Health_Statistics/NCHS/dataset_documentation/NHAMCS/stata/ed2019-stata.zip",
               "nhamcs_ed2019_stata.zip")
NH_COLS = ["AGE", "SEX", "IMMEDR", "PULSE", "RESPR", "BPSYS", "POPCT", "TEMPF", "PAYTYPER",
           "ADMITHOS", "OBSHOS", "OBSDIS", "OBSSTAY", "LOS", "PATWT"]
with zipfile.ZipFile(nh_zip) as z:
    nh = pd.read_stata(z.open("ED2019-stata.dta"), columns=NH_COLS, convert_categoricals=False)

def nh_label(r):
    if r.OBSHOS == 1: return "observation -> inpatient"
    if r.OBSDIS == 1: return "observation -> discharged"
    if r.ADMITHOS == 1: return "inpatient"
    return "discharged / other"

nh["label"] = nh.apply(nh_label, axis=1)
summary = nh.groupby("label").agg(visits=("label", "size"), national_estimate=("PATWT", "sum"))
summary["national_estimate"] = summary.national_estimate.round(-3).astype(int)
log("NHAMCS 2019 ED", "open", f"{len(nh):,} ED visits with observation outcomes")
summary

OK  NHAMCS 2019 ED: 19,481 ED visits with observation outcomes


,visits,national_estimate
label,,
discharged / other,17050,131154000
inpatient,1893,15635000
observation -> discharged,376,2689000
observation -> inpatient,162,1171000


## 3. MTSamples: sample clinical notes (Kaggle, CC0)

About 5,000 transcribed sample reports. The Kaggle download link works without logging in. These are the specialties most relevant to status review:

In [13]:
mt_zip = fetch("https://www.kaggle.com/api/v1/datasets/download/tboyle10/medicaltranscriptions", "mtsamples.zip")
mt = pd.read_csv(zipfile.ZipFile(mt_zip).open("mtsamples.csv"))
mt["medical_specialty"] = mt.medical_specialty.str.strip()

relevant = mt[mt.medical_specialty.str.contains("Emergency|History and Phy|Discharge Summary", regex=True)]
log("MTSamples", "open (CC0)", f"{len(mt):,} notes; {len(relevant):,} ED / H&P / discharge notes")
relevant.medical_specialty.value_counts().rename("notes").to_frame()

OK  MTSamples: 4,999 notes; 699 ED / H&P / discharge notes


,notes
medical_specialty,
Consult - History and Phy.,516
Discharge Summary,108
Emergency Room Reports,75


In [14]:
er = relevant[relevant.medical_specialty.str.contains("Emergency")]
example = er[er.transcription.str.contains("HISTORY OF PRESENT ILLNESS", na=False)].iloc[0]
print(example.description, "\n")
print(example.transcription[:900], "...")

 Patient has had multiple problems with his teeth due to extensive dental disease and has had many of his teeth pulled, now complains of new tooth pain to both upper and lower teeth on the left side for approximately three days.. 

CHIEF COMPLAINT:,  Toothache.,HISTORY OF PRESENT ILLNESS:  ,This is a 29-year-old male who has had multiple problems with his teeth due to extensive dental disease and has had many of his teeth pulled.  Complains of new tooth pain.  The patient states his current toothache is to both upper and lower teeth on the left side for approximately three days.  The patient states that he would have gone to see his regular dentist but he has missed so many appointments that they now do not allow him to schedule regular appointments, he has to be on standby appointments only.  The patient denies any other problems or complaints.  The patient denies any recent illness or injuries.  The patient does have OxyContin and Vicodin at home which he uses for his knee pain but h

## 4. Asclepius synthetic clinical notes (Hugging Face, CC-BY-NC-SA 4.0)

The dataset has about 158k synthetic discharge summaries. Here we only pull a few rows through the Hugging Face API to show access. For the full file, use `pandas.read_csv("hf://datasets/starmpcc/Asclepius-Synthetic-Clinical-Notes/synthetic.csv")` (needs `huggingface_hub`) or the `datasets` library.

In [15]:
HF = "https://datasets-server.huggingface.co"
ds = {"dataset": "starmpcc/Asclepius-Synthetic-Clinical-Notes"}
size = requests.get(f"{HF}/size", params=ds, timeout=60).json()["size"]["dataset"]["num_rows"]
rows = requests.get(f"{HF}/rows", params={**ds, "config": "default", "split": "train", "offset": 0, "length": 3},
                    timeout=60).json()["rows"]
asclepius = pd.DataFrame([r["row"] for r in rows])
log("Asclepius synthetic notes", "open (non-commercial)", f"{size:,} rows available")
print(asclepius.note.iloc[0][:700], "...")

OK  Asclepius synthetic notes: 158,114 rows available
Discharge Summary:

Patient: 60-year-old male with moderate ARDS from COVID-19

Hospital Course:

The patient was admitted to the hospital with symptoms of fever, dry cough, and dyspnea. During physical therapy on the acute ward, the patient experienced coughing attacks that induced oxygen desaturation and dyspnea with any change of position or deep breathing. To avoid rapid deterioration and respiratory failure, a step-by-step approach was used for position changes. The breathing exercises were adapted to avoid prolonged coughing and oxygen desaturation, and with close monitoring, the patient managed to perform strength and walking exercises at a low level. Exercise progression was low init ...


## 5. Rule text and codes for the rules engine (Ideas 2–3)

### 5a. 42 CFR 412.3, the Two-Midnight Rule, from the official eCFR API

In [16]:
titles = requests.get("https://www.ecfr.gov/api/versioner/v1/titles.json", timeout=60).json()["titles"]
as_of = next(t for t in titles if t["number"] == 42)["up_to_date_as_of"]
xml = requests.get(f"https://www.ecfr.gov/api/versioner/v1/full/{as_of}/title-42.xml",
                   params={"part": "412", "section": "412.3"}, timeout=60).text
rule_text = re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", xml))
two_midnight = re.search(r"\(d\)\(1\)[^.]*midnights\.", rule_text).group(0)
log("eCFR 42 CFR 412.3", "open API", f"current as of {as_of}")
print(two_midnight)

OK  eCFR 42 CFR 412.3: current as of 2026-10-21
(d)(1) Except as specified in paragraphs (d)(2) and (3) of this section, an inpatient admission is generally appropriate for payment under Medicare Part A when the admitting physician expects the patient to require hospital care that crosses two midnights.


### 5b. Medicare Benefit Policy Manual, Chapter 1 (inpatient hospital services)

In [17]:
from pypdf import PdfReader

bpm = PdfReader(fetch("https://www.cms.gov/Regulations-and-Guidance/Guidance/Manuals/downloads/bp102c01.pdf",
                      "cms_benefit_policy_manual_ch1.pdf"))
bpm_text = " ".join(page.extract_text() or "" for page in bpm.pages)
hit = re.search(r"[^.]*two midnights[^.]*\.", bpm_text, flags=re.I)
log("CMS Benefit Policy Manual Ch. 1", "open PDF", f"{len(bpm.pages)} pages")
print(re.sub(r"\s+", " ", hit.group(0)).strip() if hit else bpm_text[:500])

OK  CMS Benefit Policy Manual Ch. 1: 56 pages
Generally, a patient is considered an inpatient if formally admitted as inpatient with the expectation that he or she will require hospital care that is expected to span at least two midnights and occupy a bed even though it later develops that the patient can be discharged or transferred to another hospital and not actually use a hospital bed overnight.


### 5c. Inpatient Only (IPO) list: OPPS Addendum E, CY2026 final rule
Procedures on this list are inpatient regardless of expected length of stay. CMS began phasing the list out in CY2026. **The descriptors are AMA CPT text (copyrighted), so don't commit this file to GitHub.** Keep it in `data/`.

In [18]:
opps = fetch("https://www.cms.gov/files/zip/2026-nfrm-opps-addenda.zip", "cms_opps_2026_addenda.zip")
with zipfile.ZipFile(opps) as z:
    add_e = next(n for n in z.namelist() if "Addendum E" in n and n.endswith(".xlsx"))
    ipo = pd.read_excel(z.open(add_e), header=2, dtype=str).dropna(subset=["HCPCS Code"])
ipo.columns = ipo.columns.str.strip()
log("CMS Inpatient Only list", "open", f"{len(ipo):,} HCPCS codes for CY2026")

def is_inpatient_only(code):
    return code in set(ipo["HCPCS Code"].str.strip())

print("33533 (coronary artery bypass, arterial) inpatient-only?", is_inpatient_only("33533"))
ipo.head(5)

OK  CMS Inpatient Only list: 1,438 HCPCS codes for CY2026
33533 (coronary artery bypass, arterial) inpatient-only? True


,HCPCS Code,Short Descriptor,CI,SI
0,00176,Anes intraoral px rad surg,NaN,C
1,00211,Anes icr px crnec/crnot hmtm,NaN,C
2,00214,Anes icr px burr holes,NaN,C
3,00215,Anes icr px crnop/elvtn fx,NaN,C
4,00524,Anes clsd ch px pneumocents,NaN,C


### 5d. ICD-10-CM lookup (NLM Clinical Tables API, no key)

In [19]:
def icd10_search(term, n=5):
    r = requests.get("https://clinicaltables.nlm.nih.gov/api/icd10cm/v3/search",
                     params={"sf": "code,name", "terms": term, "maxList": n}, timeout=60).json()
    return pd.DataFrame(r[3], columns=["code", "name"])

log("ICD-10-CM (NLM API)", "open API", "search works")
icd10_search("sepsis")

OK  ICD-10-CM (NLM API): search works


,code,name
0,A02.1,Salmonella sepsis
1,A22.7,Anthrax sepsis
2,A26.7,Erysipelothrix sepsis
3,A32.7,Listerial sepsis
4,A42.7,Actinomycotic sepsis


## 6. Full MIMIC-IV (credentialed): how to plug it in

The demo is too small for reliable metrics, and it has no clinical notes. **MIMIC-IV-Note** (discharge summaries, radiology reports) is credentialed only. Full MIMIC-IV / MIMIC-IV-ED / MIMIC-IV-Note require that **each person** who uses them:
1. completes CITI *"Data or Specimens Only Research"* training,
2. is credentialed on PhysioNet, and
3. signs the data use agreement on each dataset's page.

The agreement prohibits sharing the data with anyone who hasn't done this, so the mentor can't download it for the team. Once credentialed, download into `data/full/` (check each page for the latest version), for example:

```
wget -r -N -c -np --user YOUR_PHYSIONET_USERNAME --ask-password https://physionet.org/files/mimiciv/<version>/
```

Then change the base paths in Section 1 to the local files; the rest of the code stays the same. **Do not send credentialed data to cloud LLM APIs** (PhysioNet's guidance requires zero retention, no training, and no human review). Use a local model, e.g. Ollama.

In [20]:
full = DATA / "full"
found = sorted(p.name for p in full.glob("*")) if full.exists() else []
print("Full MIMIC found locally:", found or "not yet (credentialed download)")

Full MIMIC found locally: not yet (credentialed download)


## Access summary

In [21]:
pd.DataFrame(STATUS)

,dataset,access,result
0,MIMIC-IV Demo,open,"100 patients, 275 hospital admissions, 107,727 lab results"
1,MIMIC-IV-ED Demo,open,"222 ED stays, 222 triage records, 1,038 ED vital-sign rows"
2,MIMIC-IV status labels,open,210 labeled ED stays; 87 candidate conversions
3,NHAMCS 2019 ED,open,"19,481 ED visits with observation outcomes"
4,MTSamples,open (CC0),"4,999 notes; 699 ED / H&P / discharge notes"
5,Asclepius synthetic notes,open (non-commercial),"158,114 rows available"
6,eCFR 42 CFR 412.3,open API,current as of 2026-10-21
7,CMS Benefit Policy Manual Ch. 1,open PDF,56 pages
8,CMS Inpatient Only list,open,"1,438 HCPCS codes for CY2026"
9,ICD-10-CM (NLM API),open API,search works
